In [ ]:
from intake_esgf import ESGFCatalog
import glob
import numpy as np
import xarray as xr
import xesmf as xe

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')

def clim(ds):
    return ds.groupby('time.month').mean('time')


def global_mean(data, lat='lat', lon='lon'):
    weights_lat_model = np.cos(np.deg2rad(data.lat))
    try:
        mean = data.weighted(weights_lat_model).mean([lat, lon])
    except ValueError:
        mean = data.weighted(weights_lat_model).mean([lat])
    return mean


def shift_lon(ds, lon_name = 'lon'):
    # Adjust lon values to make sure they are within (-180, 180)
    ds['_longitude_adjusted'] = xr.where(
        ds[lon_name] > 180,
        ds[lon_name] - 360,
        ds[lon_name])

    # reassign the new coords to as the main lon coords
    # and sort DataArray using new coordinate values
    ds = (
        ds
        .swap_dims({lon_name: '_longitude_adjusted'})
        .sel(**{'_longitude_adjusted': sorted(ds._longitude_adjusted)})
        .drop_vars(lon_name))

    ds = ds.rename({'_longitude_adjusted': lon_name})
    return ds

def calculate_nino34_index(ds):
    # ds needs to be tos-anomalies
    # 170° - 120°W -> 190 - 240°E
    nino = ds.sel(lon=slice(190,240), lat=slice(-5,5))
    nino_gm = global_mean(nino).compute()
    # nino_gm = nino.mean(['lon', 'lat']).compute()
    return nino_gm

def calculate_nao(ds):
    ds_south = ds.sel(lat=slice(20,55), lon=slice(-90,60))
    ds_south = global_mean(ds_south)
    # ds_south = ds_south.mean(['lat', 'lon'])
    ds_north = ds.sel(lat=slice(55,90), lon=slice(-90,60))
    ds_north = global_mean(ds_north)
    # ds_north = ds_north.mean(['lat', 'lon'])
    nao = (ds_south - ds_north).compute()
    nao = nao.to_dataset(name='zg')
    return nao


def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    return ds_djf


def regrid(in_grid,out_grid, var): 
    reg_bil = xe.Regridder(in_grid,out_grid, "bilinear")
    var_bil = reg_bil(in_grid[var])
    return var_bil



In [4]:

outdir = "/sto4/data/Intermediate/VolMIP/"
models = ['CanESM5', 'MIROC-ES2L', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'UKESM1-0-LL']
member_id = ['r1i1p2f1', 'r1i1p1f2', 'r1i1p1f1', 'r1i1p1f2', 'r1i1p1f2']

In [5]:
# cat=ESGFCatalog()
# metagrid_search = cat.search(activity_id='VolMIP', experiment_id='volc-pinatubo-full', variable_id=['utendepfd'], table_id='EmonZ',source_id=models[0], latest=True)
# ds = metagrid_search.to_path_dict()

In [6]:
# open ESGF catalog
cat=ESGFCatalog()
metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id='IPSL-CM6A-LR',
        variable_id=['ta','ua','tas'], latest=True, table_id=['Amon'], member_id='r1i1p1f1')
ds = metagrid_search.to_path_dict()

   Searching indices:   0%|          |0/1 [       ?index/s]

Get file information:   0%|          |0/1 [       ?index/s]

In [7]:
ds

{'ta': [PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/piControl/r1i1p1f1/Amon/ta/gr/v20200326/ta_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_305001-354912.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/piControl/r1i1p1f1/Amon/ta/gr/v20200326/ta_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_355001-384912.nc')],
 'tas': [PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/piControl/r1i1p1f1/Amon/tas/gr/v20200326/tas_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_305001-354912.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/piControl/r1i1p1f1/Amon/tas/gr/v20200326/tas_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_355001-384912.nc')],
 'ua': [PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/piControl/r1i1p1f1/Amon/ua/gr/v20200326/ua_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_305001-354912.nc'),
  PosixPath('/sto0/user/Kperny/.esgf/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/piControl/r1i1p1f1/Amon/ua/gr/v20200326/ua_Amon_IPSL-CM6A-LR_piC

In [ ]:
# model = models[0]
# path = outdir + 'pinatubo-full/' + model + '/full/'
# time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
# for file in glob.glob(path + 'zg*.nc'):
#     print(file.split('/')[-1].split('_')[4])
    # data = xr.open_dataset(file, decode_times=time_coder)
    # ds_prep = shift_lon(data.sel(plev=50000, lat=slice(0,100)))
    # nao = calculate_nao(ds_prep.zg)
    # # write output
    # nao.to_netcdf(outdir + 'pinatubo-full/' + model + '/NAO_' + model + '_pinatubo-full_' + ens + '.nc')

In [ ]:
for model, id in zip(models, member_id):
    cat=ESGFCatalog()
    metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
           variable_id=['utendepfd'], latest=True, member_id=id, grid_label='gn')
    print(metagrid_search)
    #ds = metagrid_search.to_path_dict()

In [ ]:
metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', variable_id=['utendepfd'], latest=True, source_id=models[0], table_id='EmonZ')
metagrid_search
ds = metagrid_search.to_path_dict()
ds

   Searching indices:   0%|          |0/1 [       ?index/s]

Summary information for 1 results:
mip_era               [CMIP6]
activity_drs           [CMIP]
institution_id        [CCCma]
source_id           [CanESM5]
experiment_id     [piControl]
member_id          [r1i1p2f1]
table_id              [EmonZ]
variable_id       [utendepfd]
grid_label               [gn]
dtype: object

In [ ]:
model = 'IPSL-CM6A-LR'
metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
            variable_id=['tos', 'zg'], latest=True, table_id=['Omon','Amon'])
metagrid_search

In [ ]:
## DOWNLOAD ZG AND TOS FOR PICONTROL 
outdir = "/sto4/data/Intermediate/VolMIP/"
models = ['CanESM5', 'MIROC-ES2L', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'UKESM1-0-LL', 'IPSL-CM6A-LR']
# according to member id in VOLMIP!!
member_id = ['r1i1p2f1', 'r1i1p1f2', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2', 'r1i1p1f1']

for model, id in zip(models, member_id):
    print(model, id)
    # open ESGF catalog
    cat=ESGFCatalog()
    metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
            variable_id=['tos', 'zg'], latest=True, table_id=['Omon','Amon'], member_id=id)
    print(metagrid_search)